In [ ]:
%pip install -U "kagglehub[pandas-datasets]"


In [ ]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

np.random.seed(42)

file_path = 'training.1600000.processed.noemoticon.csv'

df_raw = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'kazanova/sentiment140',
    file_path,
    pandas_kwargs={
        'encoding': 'latin-1',
        'header': None,
        'on_bad_lines': 'skip',
    },
)

df_raw.head()

In [ ]:
df = df_raw.copy()
df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']
df['target'] = df['target'].replace(4,1)
df = df[['text', 'target']]
df.head()

In [ ]:
import re

def clean_tweet(text):
    text = text.lower()  # lowercase

    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'@\w+', '', text)     # remove mentions
    text = re.sub(r'#\w+', '', text)     # remove hashtags

    text = re.sub(r'[^a-z\s]', '', text) # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces

    return text

In [ ]:
df['clean_text'] = df['text'].apply(clean_tweet)
df[['text', 'clean_text']].head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
X_train_tfidf.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)

model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import re

EMOTICONS = [
    (r'(?<!\w):-?\)(?!\w)', 'smile'),
    (r'(?<!\w):-?\((?!\w)', 'sad'),
    (r'(?<!\w):d(?!\w)',    'laugh'),
    (r'(?<!\w)<3(?!\w)',    'love'),
    (r'(?<!\w);-?\)(?!\w)', 'wink'),
]

def replace_emoticons(text):
    for pattern, word in EMOTICONS:
        text = re.sub(pattern, f' {word} ', text)
    return text

def clean_tweet_advanced(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r'#(\w+)', r' \1 ', text)
    text = replace_emoticons(text)          # ← replaces the 6 bare lines
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df['clean_text_advanced'] = df['text'].apply(clean_tweet_advanced)
df[['text', 'clean_text', 'clean_text_advanced']].head(10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df['clean_text_advanced']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Controlled Comparison Task List

This section completes the remaining proposal work on a fixed stratified split. It does four things:

- Standardizes a shared 80/10/10 train/validation/test split for every model.
- Tunes Logistic Regression, linear SVM, and an MLP neural baseline on the validation split.
- Evaluates each model on the full held-out test set and on a noisy Twitter-specific subset.
- Produces runtime, throughput, confusion matrices, and error-analysis tables for the final report.


In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TRAIN_FRACTION = 0.80
VALIDATION_FRACTION = 0.10
TEST_FRACTION = 0.10
MAX_FEATURES = 5000
EXPERIMENT_SAMPLE_FRACTION = 1.0  # Lower this for faster notebook iteration if needed.
MODEL_MAX_TRAIN_ROWS = {
    'Logistic Regression': None,
    'Linear SVM': None,
    'MLP Neural Baseline': 100000,
}

if 'clean_text_advanced' not in df.columns:
    raise ValueError('Run the advanced preprocessing cells before the controlled comparison section.')

analysis_df = df[['text', 'clean_text_advanced', 'target']].dropna().copy()
analysis_df['text'] = analysis_df['text'].astype(str)
analysis_df['clean_text_advanced'] = analysis_df['clean_text_advanced'].astype(str)

if not 0 < EXPERIMENT_SAMPLE_FRACTION <= 1.0:
    raise ValueError('EXPERIMENT_SAMPLE_FRACTION must be in the interval (0, 1].')

if EXPERIMENT_SAMPLE_FRACTION < 1.0:
    analysis_df = (
        analysis_df
        .groupby('target', group_keys=False)
        .sample(frac=EXPERIMENT_SAMPLE_FRACTION, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

train_df, holdout_df = train_test_split(
    analysis_df,
    test_size=VALIDATION_FRACTION + TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=analysis_df['target']
)

val_df, test_df = train_test_split(
    holdout_df,
    test_size=TEST_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION),
    random_state=RANDOM_STATE,
    stratify=holdout_df['target']
)

train_val_df = pd.concat([train_df, val_df], ignore_index=True)

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train_df), 'positive_rate': train_df['target'].mean()},
    {'split': 'validation', 'rows': len(val_df), 'positive_rate': val_df['target'].mean()},
    {'split': 'test', 'rows': len(test_df), 'positive_rate': test_df['target'].mean()},
])

split_summary


In [ ]:
from time import perf_counter
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
import warnings

warnings.filterwarnings('ignore', category=ConvergenceWarning)

EMOTICON_PATTERN = re.compile(r'(:\)|:-\)|:\(|:-\(|:d|<3|;\)|;\-\))', re.IGNORECASE)
EMOJI_PATTERN = re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF]+')
REPEATED_CHAR_PATTERN = re.compile(r'(.)\1{2,}')
SLANG_PATTERN = re.compile(r'\b(lol|omg|wtf|idk|smh|tbh|imo|btw|lmao|rofl|ikr|u|ur|ya|tho|gonna|wanna)\b', re.IGNORECASE)
NEGATION_PATTERN = re.compile(r"\b(no|not|never|can't|cannot|don't|didn't|isn't|wasn't|won't|shouldn't|couldn't)\b", re.IGNORECASE)
SARCASTIC_PATTERN = re.compile(r'\b(yeah right|as if|sure jan)\b', re.IGNORECASE)

NOISE_COLUMNS = [
    'has_url',
    'has_mention',
    'has_hashtag',
    'has_repeated_chars',
    'has_emoticon',
    'has_emoji',
    'has_slang',
]

ERROR_TAG_COLUMNS = [
    'has_negation',
    'has_sarcasm_marker',
    'has_all_caps',
    'has_excess_punctuation',
]

def add_noise_and_error_flags(frame):
    tagged = frame.copy()
    tagged['has_url'] = tagged['text'].str.contains(r'http\S+|www\.\S+', regex=True, na=False)
    tagged['has_mention'] = tagged['text'].str.contains(r'@\w+', regex=True, na=False)
    tagged['has_hashtag'] = tagged['text'].str.contains(r'#\w+', regex=True, na=False)
    tagged['has_repeated_chars'] = tagged['text'].apply(lambda value: bool(REPEATED_CHAR_PATTERN.search(str(value))))
    tagged['has_emoticon'] = tagged['text'].apply(lambda value: bool(EMOTICON_PATTERN.search(str(value))))
    tagged['has_emoji'] = tagged['text'].apply(lambda value: bool(EMOJI_PATTERN.search(str(value))))
    tagged['has_slang'] = tagged['text'].apply(lambda value: bool(SLANG_PATTERN.search(str(value))))
    tagged['noise_count'] = tagged[NOISE_COLUMNS].sum(axis=1)
    tagged['is_noisy'] = tagged['noise_count'] > 0
    tagged['has_negation'] = tagged['text'].apply(lambda value: bool(NEGATION_PATTERN.search(str(value))))
    tagged['has_sarcasm_marker'] = tagged['text'].apply(lambda value: bool(SARCASTIC_PATTERN.search(str(value))))
    tagged['has_all_caps'] = tagged['text'].str.contains(r'\b[A-Z]{3,}\b', regex=True, na=False)
    tagged['has_excess_punctuation'] = tagged['text'].str.contains(r'[!?]{2,}', regex=True, na=False)
    return tagged

def build_vectorizer(vectorizer_config):
    return TfidfVectorizer(
        max_features=vectorizer_config.get('max_features', MAX_FEATURES),
        min_df=vectorizer_config['min_df'],
        ngram_range=vectorizer_config['ngram_range'],
        sublinear_tf=True,
    )

def sample_training_frame(frame, model_name):
    max_rows = MODEL_MAX_TRAIN_ROWS.get(model_name)
    if max_rows is None or len(frame) <= max_rows:
        return frame.reset_index(drop=True)

    sampled_frame, _ = train_test_split(
        frame,
        train_size=max_rows,
        random_state=RANDOM_STATE,
        stratify=frame['target'],
    )
    return sampled_frame.reset_index(drop=True)

def build_model(model_name, model_config):
    if model_name == 'Logistic Regression':
        return LogisticRegression(
            C=model_config['C'],
            max_iter=1000,
            solver='liblinear',
            random_state=RANDOM_STATE,
        )
    if model_name == 'Linear SVM':
        return LinearSVC(
            C=model_config['C'],
            dual=False,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    if model_name == 'MLP Neural Baseline':
        return MLPClassifier(
            hidden_layer_sizes=model_config['hidden_layer_sizes'],
            alpha=model_config['alpha'],
            learning_rate_init=model_config['learning_rate_init'],
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=3,
            max_iter=20,
            batch_size='auto',
            random_state=RANDOM_STATE,
        )
    raise ValueError(f'Unsupported model: {model_name}')

def get_scores(model, features):
    if hasattr(model, 'decision_function'):
        return model.decision_function(features)
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(features)[:, 1]
    raise ValueError('Model must expose decision_function or predict_proba for ROC-AUC.')

def summarize_metrics(y_true, y_pred, y_score):
    roc_auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float('nan')
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc,
    }

def parameter_count(model):
    if hasattr(model, 'coef_'):
        return int(model.coef_.size + model.intercept_.size)
    if hasattr(model, 'coefs_'):
        return int(sum(arr.size for arr in model.coefs_) + sum(arr.size for arr in model.intercepts_))
    return None

test_df = add_noise_and_error_flags(test_df)
noisy_test_df = test_df[test_df['is_noisy']].copy()

noise_summary = (
    test_df[NOISE_COLUMNS + ['noise_count', 'is_noisy']]
    .mean(numeric_only=True)
    .sort_values(ascending=False)
    .rename('share_of_test_set')
)

print(f'Noisy subset size: {len(noisy_test_df):,} / {len(test_df):,}')
noise_summary


In [ ]:
vectorizer_grid = {
    'Logistic Regression': [
        {'ngram_range': (1, 1), 'min_df': 5, 'max_features': MAX_FEATURES},
        {'ngram_range': (1, 2), 'min_df': 5, 'max_features': MAX_FEATURES},
    ],
    'Linear SVM': [
        {'ngram_range': (1, 1), 'min_df': 5, 'max_features': MAX_FEATURES},
        {'ngram_range': (1, 2), 'min_df': 5, 'max_features': MAX_FEATURES},
    ],
    'MLP Neural Baseline': [
        {'ngram_range': (1, 1), 'min_df': 5, 'max_features': 3000},
    ],
}

model_grids = {
    'Logistic Regression': {
        'C': [0.5, 1.0, 2.0],
    },
    'Linear SVM': {
        'C': [0.5, 1.0, 2.0],
    },
    'MLP Neural Baseline': {
        'hidden_layer_sizes': [(64,), (128,)],
        'alpha': [1e-3, 1e-2],
        'learning_rate_init': [5e-4],
    },
}

validation_rows = []
best_configs = {}

for model_name, model_grid in model_grids.items():
    model_best_row = None
    model_train_df = sample_training_frame(train_df, model_name)

    for vectorizer_config in vectorizer_grid[model_name]:
        vectorizer = build_vectorizer(vectorizer_config)
        X_train = vectorizer.fit_transform(model_train_df['clean_text_advanced'])
        X_val = vectorizer.transform(val_df['clean_text_advanced'])

        for model_config in ParameterGrid(model_grid):
            model = build_model(model_name, model_config)

            train_start = perf_counter()
            model.fit(X_train, model_train_df['target'])
            train_time = perf_counter() - train_start

            score_start = perf_counter()
            val_pred = model.predict(X_val)
            val_score = get_scores(model, X_val)
            inference_time = perf_counter() - score_start

            row = {
                'model': model_name,
                'ngram_range': vectorizer_config['ngram_range'],
                'min_df': vectorizer_config['min_df'],
                'max_features': vectorizer_config.get('max_features', MAX_FEATURES),
                'training_rows': len(model_train_df),
                'train_time_sec': train_time,
                'validation_inference_sec': inference_time,
                'params': parameter_count(model),
            }
            row.update(model_config)
            row.update(summarize_metrics(val_df['target'], val_pred, val_score))
            validation_rows.append(row)

            if model_best_row is None or row['f1'] > model_best_row['f1']:
                model_best_row = row.copy()

    best_configs[model_name] = model_best_row

validation_results = pd.DataFrame(validation_rows).sort_values(['model', 'f1', 'roc_auc'], ascending=[True, False, False])
best_config_summary = pd.DataFrame(best_configs.values()).sort_values('f1', ascending=False).reset_index(drop=True)

best_config_summary


In [ ]:
final_models = {}
comparison_rows = []
prediction_store = {}
score_store = {}
report_store = {}
confusion_store = {}

for model_name, best_config in best_configs.items():
    model_train_val_df = sample_training_frame(train_val_df, model_name)
    vectorizer_config = {
        'ngram_range': tuple(best_config['ngram_range']),
        'min_df': int(best_config['min_df']),
        'max_features': int(best_config['max_features']),
    }
    model_config = {key: best_config[key] for key in best_config.keys() if key in {'C', 'hidden_layer_sizes', 'alpha', 'learning_rate_init'}}

    vectorizer = build_vectorizer(vectorizer_config)
    X_train_val = vectorizer.fit_transform(model_train_val_df['clean_text_advanced'])
    model = build_model(model_name, model_config)

    train_start = perf_counter()
    model.fit(X_train_val, model_train_val_df['target'])
    train_time = perf_counter() - train_start

    final_models[model_name] = {
        'vectorizer': vectorizer,
        'model': model,
        'config': {**vectorizer_config, **model_config},
        'parameter_count': parameter_count(model),
    }

    for split_name, frame in [('full_test', test_df), ('noisy_test', noisy_test_df)]:
        if frame.empty:
            continue

        X_eval = vectorizer.transform(frame['clean_text_advanced'])

        predict_start = perf_counter()
        y_pred = model.predict(X_eval)
        prediction_time = perf_counter() - predict_start

        score_start = perf_counter()
        y_score = get_scores(model, X_eval)
        scoring_time = perf_counter() - score_start

        metrics = summarize_metrics(frame['target'], y_pred, y_score)
        metrics.update({
            'model': model_name,
            'split': split_name,
            'rows': len(frame),
            'training_rows': len(model_train_val_df),
            'train_time_sec': train_time,
            'prediction_time_sec': prediction_time,
            'scoring_time_sec': scoring_time,
            'tweets_per_second': len(frame) / prediction_time if prediction_time > 0 else float('inf'),
            'parameter_count': final_models[model_name]['parameter_count'],
        })
        comparison_rows.append(metrics)

        if split_name == 'full_test':
            prediction_store[model_name] = y_pred
            score_store[model_name] = y_score
            report_store[model_name] = classification_report(frame['target'], y_pred, digits=4)
            confusion_store[model_name] = confusion_matrix(frame['target'], y_pred)

comparison_results = pd.DataFrame(comparison_rows).sort_values(['split', 'f1', 'roc_auc'], ascending=[True, False, False]).reset_index(drop=True)

comparison_results[['model', 'split', 'rows', 'training_rows', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'train_time_sec', 'prediction_time_sec', 'tweets_per_second', 'parameter_count']]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for model_name, report in report_store.items():
    print(model_name)
    print(report)
    print('-' * 80)

fig, axes = plt.subplots(1, len(confusion_store), figsize=(5 * len(confusion_store), 4))
if len(confusion_store) == 1:
    axes = [axes]

for ax, (model_name, cm) in zip(axes, confusion_store.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(model_name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


In [ ]:
def build_error_table(frame, y_pred, y_score, model_name):
    error_frame = frame.copy().reset_index(drop=True)
    error_frame['prediction'] = y_pred
    error_frame['score'] = y_score
    error_frame['model'] = model_name
    error_frame['error_type'] = np.where(
        (error_frame['target'] == 0) & (error_frame['prediction'] == 1),
        'false_positive',
        np.where(
            (error_frame['target'] == 1) & (error_frame['prediction'] == 0),
            'false_negative',
            'correct'
        )
    )
    return error_frame[error_frame['error_type'] != 'correct'].copy()

error_frames = []
for model_name, y_pred in prediction_store.items():
    error_frames.append(build_error_table(test_df, y_pred, score_store[model_name], model_name))

all_errors = pd.concat(error_frames, ignore_index=True) if error_frames else pd.DataFrame()

if not all_errors.empty:
    error_category_summary = (
        all_errors
        .groupby(['model', 'error_type'])[NOISE_COLUMNS + ERROR_TAG_COLUMNS]
        .mean()
        .round(3)
    )

    representative_errors = (
        all_errors
        .assign(confidence_margin=lambda frame: np.abs(frame['score']))
        .sort_values(['model', 'confidence_margin'], ascending=[True, False])
        .groupby(['model', 'error_type'])
        .head(5)
        [['model', 'error_type', 'text', 'target', 'prediction', 'score'] + NOISE_COLUMNS + ERROR_TAG_COLUMNS]
        .reset_index(drop=True)
    )

    display(error_category_summary)
    display(representative_errors)
else:
    print('No misclassifications were captured. Re-run the evaluation cells first.')


In [ ]:
import joblib
from pathlib import Path

output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

for model_name, bundle in final_models.items():
    artifact_prefix = model_name.lower().replace(' ', '_')
    joblib.dump(bundle['vectorizer'], output_dir / f'{artifact_prefix}_vectorizer.pkl')
    joblib.dump(bundle['model'], output_dir / f'{artifact_prefix}_model.pkl')

split_summary.to_csv(output_dir / 'split_summary.csv', index=False)
best_config_summary.to_csv(output_dir / 'best_config_summary.csv', index=False)
validation_results.to_csv(output_dir / 'validation_results.csv', index=False)
comparison_results.to_csv(output_dir / 'comparison_results.csv', index=False)
noise_summary.rename_axis('feature').reset_index().to_csv(output_dir / 'noise_summary.csv', index=False)

if 'error_category_summary' in globals():
    error_category_summary.reset_index().to_csv(output_dir / 'error_category_summary.csv', index=False)
if 'representative_errors' in globals():
    representative_errors.to_csv(output_dir / 'representative_errors.csv', index=False)

print('Saved tuned model artifacts and report tables to the outputs/ directory.')


## Final Evaluation Writeup

The final comparison used a fixed stratified 80/10/10 train/validation/test split with 1,280,000 training tweets, 160,000 validation tweets, and 160,000 held-out test tweets. Logistic Regression and linear SVM were tuned on the full training split with TF-IDF features up to 5,000 dimensions, while the MLP neural baseline was tuned on a capped stratified sample of 100,000 training tweets with a 3,000-feature unigram TF-IDF representation so the neural section remained computationally feasible.

On the full test split, the strongest model by F1-score was the linear SVM with `F1 = 0.800978`, followed very closely by Logistic Regression with `F1 = 0.800764`. Logistic Regression produced the best ROC-AUC at `0.878590`, while linear SVM reached `0.878336`. The MLP neural baseline trailed both classical models with `F1 = 0.778599` and `ROC-AUC = 0.865595`. This means the classical TF-IDF baselines remained the most effective models in this project, and the added neural capacity of the MLP did not translate into better generalization on Sentiment140 under the shared evaluation protocol.

The practical trade-off is also clear in the runtime metrics. Logistic Regression trained in about `6.22` seconds on the final train+validation split and linear SVM trained in about `8.90` seconds, while the MLP required about `9.75` seconds even though it used far fewer training rows. During inference, the linear models processed roughly `58-65 million` tweets per second in the notebook timing measurements, while the MLP processed about `2.23 million` tweets per second on the full test split. Within this setup, the classical models were both more accurate and more efficient.

Robustness to noisy Twitter artifacts was evaluated on a noisy subset of 106,942 held-out tweets. The noisy subset did not reduce performance for any model. Linear SVM improved from `0.800978` F1 on the full test split to `0.809168` on the noisy subset, Logistic Regression improved from `0.800764` to `0.808697`, and the MLP improved from `0.778599` to `0.786780`. This suggests that the selected noise markers were not purely adversarial for sentiment classification after preprocessing; many noisy tweets still preserved strong sentiment cues, and the normalization pipeline may have removed some of the surface variation that would otherwise make them harder.


## Error Analysis Method, Findings, and Success Criteria

Error analysis focused on false positives and false negatives from the held-out test set, then grouped those errors by interpretable linguistic markers. Across all three models, user mentions and repeated characters were common in both error types, which is expected because those artifacts are frequent in Twitter text overall. The more informative pattern was that negation appeared much more often in false negatives than in false positives for the classical models. For example, negation was present in about `29%` of linear-model false negatives but only about `17%` of false positives. This supports the claim that sentiment can still be difficult when the local wording contains negative tokens even though the overall tweet label is positive.

Representative false negatives show the main weakness clearly. Tweets such as `no sad faces! only smiles`, `unfortunately i cannot. sorry. #badoptus`, and `Got the WORST migraine... I no but it hurt!` were labeled positive in the dataset but predicted as negative by the models because they contain strong surface-negative words like `sad`, `unfortunately`, `sorry`, `worst`, and `hurt`. Representative false positives show the opposite problem: expressions such as `thanks`, `your welcome`, `love`, or `good morning` often pushed the models toward the positive class even when the distant-supervision label was negative. Some of these are likely annotation-noise cases from Sentiment140 rather than clean model failures.

The results therefore answer the project question in a specific way: under this preprocessing and TF-IDF setup, classical linear models were already strong enough to handle the selected Twitter-specific noise patterns, and the MLP neural baseline did not provide a measurable robustness advantage. The notebook still satisfies the proposal's minimum success criteria because it includes tuned Logistic Regression and linear SVM baselines, at least one tuned neural model, shared-split evaluation with accuracy, precision, recall, F1-score, ROC-AUC, confusion matrices, and a direct comparison between the full test set and a noisy Twitter-specific subset.


## Resources, Contributions, and Final Conclusion

The project uses the Sentiment140 dataset loaded through `kagglehub`, advanced tweet normalization implemented in the notebook, and scikit-learn models for both classical and neural baselines. The final export step writes reusable artifacts and report tables to `outputs/`, including split summaries, validation-search results, final comparison metrics, noise summaries, and representative error tables.

Contribution status after the completed run:

- Member 1 work is represented by data loading, preprocessing, and the Logistic Regression baseline.
- Member 2 work is represented by the shared split, tuned linear SVM, robustness subset, expanded metrics, confusion matrices, and evaluation/error-analysis writeup.
- Shared work is represented by the MLP neural baseline, runtime and throughput reporting, artifact export, and the final integrated comparison across model families.

Final conclusion: the strongest models in this project were the classical TF-IDF baselines, not the neural baseline. Linear SVM achieved the best F1-score, Logistic Regression achieved the best ROC-AUC, and both clearly outperformed the MLP while also being faster. For this dataset and preprocessing pipeline, added model complexity was not necessary to obtain strong sentiment-classification performance on noisy Twitter text.
